# 2. Gather posts

For each paper picked in notebook 1, pull all Altmetric `posts` rows that mention it. Bluesky / X / Reddit / blogs / news / Wikipedia / policy etc. all flow through here.

Inputs: `output/<RUN_ID>/papers.parquet` from notebook 1.
Outputs: `output/<RUN_ID>/posts.parquet`.

See `PIPELINE_PLAN.md` § S19 / S5 / S4. Dry-run cell precedes the real run.

## Setup

Set `RUN_ID` to a run created by notebook 1 (defaults to the most recent one).

In [ ]:
import os
from pathlib import Path

os.environ.pop("GOOGLE_APPLICATION_CREDENTIALS", None)

REPO_ROOT = Path.cwd().parent.resolve()
OUTPUT_ROOT = REPO_ROOT / "altendor" / "output"

# Override RUN_ID here if you want a specific historical run instead of the latest.
RUN_ID: str | None = None
if RUN_ID is None:
    candidates = sorted([p.name for p in OUTPUT_ROOT.iterdir() if (p / "papers.parquet").exists()])
    if not candidates:
        raise FileNotFoundError("No run with papers.parquet found. Run notebook 1 first.")
    RUN_ID = candidates[-1]

OUT_DIR = OUTPUT_ROOT / RUN_ID
print(f"Using run id: {RUN_ID}\nOutput dir: {OUT_DIR}")

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq_client = bigquery.Client()

papers_df = pd.read_parquet(OUT_DIR / "papers.parquet")
ro_ids = papers_df["ro_id"].dropna().astype(str).tolist()
print(f"Loaded {len(papers_df)} papers \u2192 {len(ro_ids)} unique ro_ids")

## Fetch posts

Server-side `UNNEST(research_outputs_ids)` explodes the posts so each (post, ro_id) pair becomes one row. A single post mentioning two of our papers appears twice — that's intentional, downstream stages key off `(post_id, ro_id)`.

**Dry-run first.**

In [ ]:
from altendor.bigquery.preflight import preflight
from altendor.bigquery.queries import posts_by_research_output_ids

posts_sql, posts_cfg = posts_by_research_output_ids(ro_ids)
preflight(bq_client, posts_sql, job_config=posts_cfg)

In [ ]:
from altendor.sources.altmetric import fetch_posts_for_papers

posts_df = fetch_posts_for_papers(bq_client, ro_ids)
print(f"Fetched {len(posts_df)} post rows across {posts_df['ro_id'].nunique()} papers.")
posts_df.head(5)

## Per-platform stats

Quick distribution of post types. Bluesky (`bsky`) and Reddit (`rdt`) are the ones we'll deeply enrich (notebook 3); X (`tweet`), blogs, news (`msm`) keep `posts.title` as-is.

In [ ]:
by_type = posts_df.groupby("type").size().rename("count").sort_values(ascending=False)
by_paper = posts_df.groupby("ro_id").size().rename("posts").sort_values(ascending=False)

print("Posts per platform:")
print(by_type.to_string())
print("\nPosts per paper:")
print(by_paper.to_string())

## Write outputs

Persist `posts.parquet` and append to the run's `manifest.json`.

In [ ]:
import json

posts_path = OUT_DIR / "posts.parquet"
posts_df.to_parquet(posts_path, index=False)

manifest_path = OUT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
manifest["stage_2_gather_posts"] = {
    "n_posts": int(len(posts_df)),
    "n_papers_with_posts": int(posts_df["ro_id"].nunique()),
    "by_platform": {str(k): int(v) for k, v in by_type.items()},
}
manifest.setdefault("output_files", []).append(str(posts_path.relative_to(REPO_ROOT)))
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
print(f"Wrote {posts_path}\nUpdated {manifest_path}")